# CPE393 Text Analytics — Opinosis Lab
**Topic:** Graph-based summarization of redundant opinions  

Based on Ganesan, Zhai & Han (2010), *Opinosis: A Graph-Based Approach to Abstractive Summarization of Highly Redundant Opinions*.

**Learning objectives:** Build an Opinosis-style directed word graph, inspect positional references (SID, PID), find repeated paths, and explain how combining fragments differs from selecting complete sentences.

**Scope:** This is a deliberately simplified teaching implementation of **Algorithm 1 (graph construction)** and a small part of **Algorithm 2 (candidate path search and ranking)**. It does **not** reproduce the full Algorithm 3, POS-based validity constraints, gapped-overlap scoring, or automated collapse/stitching. The final fusion activity is guided rather than automatic.



## 1. Read the opinions (5 minutes)
These are synthetic product-review sentences created for this lab.

Predict: which opinions are repeated? Which information is not shared by most reviewers?


In [ ]:
reviews = [
    "the battery life is great",
    "the battery life is great",
    "the battery life is long",
    "the battery life is long",
    "the screen is bright",
    "the screen is bright",
]
for sid, sentence in enumerate(reviews, start=1):
    print(sid, sentence)


## 2. Build an Opinosis-style graph (15 minutes)
In the paper's Algorithm 1, each unique **word/POS unit** is a node; directed edges preserve word order; **PRI** stores each occurrence as `(SID, PID)`.

For simplicity, we use lowercase word labels instead of word/POS labels. **SID** is the sentence number and **PID** is the 1-based position.


In [ ]:
from collections import defaultdict
import networkx as nx

def build_graph(sentences):
    G = nx.DiGraph()
    for sid, sentence in enumerate(sentences, start=1):
        tokens = sentence.lower().split()
        for pid, token in enumerate(tokens, start=1):
            if token not in G:
                G.add_node(token, pri=[])
            # TODO 1: append (sid, pid) to the node PRI
            ...
            if pid > 1:
                previous = tokens[pid - 2]
                # TODO 2: add a directed edge from previous to token
                ...
    return G

G = build_graph(reviews)
print("Nodes:", list(G.nodes))
print("Edges:", list(G.edges))
print("PRI for 'battery':", G.nodes["battery"]["pri"])


**Checkpoint:** What does the PRI for `battery` tell us? Why are edges directed?

## 3. Find repeated word paths (15 minutes)
The full paper uses **positional overlap** and gapped subsequences. For this beginner exercise, a path is counted only when it occurs as a **contiguous sequence** in an input review. This is a simplified redundancy score, **not** the paper's scoring formula.

We restrict candidate paths to 3–6 words, then sort by number of supporting sentences.


In [ ]:
def supporting_sentences(path, sentences):
    phrase = list(path)
    matches = []
    for sid, sentence in enumerate(sentences, start=1):
        tokens = sentence.lower().split()
        if any(tokens[i:i+len(phrase)] == phrase
               for i in range(len(tokens)-len(phrase)+1)):
            matches.append(sid)
    return matches

def candidates(G, sentences, min_words=3, max_words=6):
    found = {}
    def walk(path):
        if len(path) >= min_words:
            support = supporting_sentences(path, sentences)
            # TODO 3: retain paths supported by at least two reviews
            ...
        if len(path) == max_words:
            return
        for nxt in G.successors(path[-1]):
            if nxt not in path:
                walk(path + [nxt])
    for node in G.nodes:
        walk([node])
    return sorted(found.items(), key=lambda x: (-len(x[1]), -len(x[0]), x[0]))

ranked = candidates(G, reviews)
for path, support in ranked[:12]:
    print(f"{len(support)} reviews | {' '.join(path)} | SID={support}")


## 4. Experiment and interpret (10 minutes)
Change one review to **"the battery life is poor"**, rebuild the graph, and inspect the repeated candidates again.

**Q1.** Which paths remain strongly supported?  
**Q2.** Why is a high redundancy score not necessarily evidence that an opinion is true?  
**Q3.** How does an Opinosis word graph differ from the sentence graph used in TextRank?


In [ ]:
modified = reviews.copy()
modified[1] = "the battery life is poor"
G2 = build_graph(modified)
for path, support in candidates(G2, modified)[:10]:
    print(len(support), " ".join(path), support)
